# SVM Model — Adult Census Dataset
**Model:** Support Vector Machine (RBF Kernel)  
**Task:** Binary classification — predict if income >50K  
**Split:** Shared pre-processed train/test split (OHE features)  
**Author:** [Your Name] | Group ML Report
---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, time
warnings.filterwarnings('ignore')

from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_auc_score, roc_curve, precision_score, recall_score,
    f1_score, ConfusionMatrixDisplay
)

print('Libraries loaded.')

## 1. Load Shared Pre-processed Data
Using the shared OHE-encoded train/test split to ensure consistency across all group models.

In [ ]:
X_train = pd.read_csv('X_train_ohe.csv')
X_test  = pd.read_csv('X_test_ohe.csv')
y_train = pd.read_csv('y_train.csv').iloc[:, 0]
y_test  = pd.read_csv('y_test.csv').iloc[:, 0]

print(f'X_train: {X_train.shape} | X_test: {X_test.shape}')
print(f'Class distribution (train):\n{y_train.value_counts()}')
print(f'\nClass ratio: {y_train.mean():.2%} positive (>50K)')

## 2. SVM Model
### Why RBF Kernel?
The RBF (Radial Basis Function) kernel is well-suited for tabular data where the decision boundary is non-linear. With 87 OHE features, linear separability is unlikely across all feature interactions.

### Hyperparameters
| Parameter | Value | Reason |
|-----------|-------|--------|
| `kernel` | `rbf` | Non-linear boundary |
| `C` | `1.0` | Default regularisation |
| `gamma` | `scale` | Auto-scales with n_features |
| `class_weight` | `balanced` | Corrects 3:1 class imbalance |
| `probability` | `True` | Enables ROC-AUC evaluation |

In [ ]:
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('svm',    SVC(kernel='rbf', C=1.0, gamma='scale',
                   class_weight='balanced', probability=True, random_state=42))
])

t0 = time.time()
pipe.fit(X_train, y_train)
train_time = time.time() - t0
print(f'Training complete in {train_time:.1f}s')

## 3. Model Evaluation

In [ ]:
y_pred = pipe.predict(X_test)
y_prob = pipe.predict_proba(X_test)[:, 1]

acc = accuracy_score(y_test, y_pred)
roc = roc_auc_score(y_test, y_prob)

print(f'Accuracy : {acc:.4f}')
print(f'ROC-AUC  : {roc:.4f}')
print(f'Precision: {precision_score(y_test, y_pred):.4f}')
print(f'Recall   : {recall_score(y_test, y_pred):.4f}')
print(f'F1 Score : {f1_score(y_test, y_pred):.4f}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=['<=50K', '>50K']))

### 3.1 Confusion Matrix

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm, display_labels=['<=50K', '>50K']).plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title('SVM — Confusion Matrix\nAdult Census Dataset', fontsize=12, pad=10)
plt.tight_layout()
plt.savefig('output_plots/svm_01_confusion_matrix.png', dpi=150)
plt.show()

### 3.2 ROC Curve

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_prob)
fig, ax = plt.subplots(figsize=(5, 4))
ax.plot(fpr, tpr, color='steelblue', lw=2, label=f'SVM RBF (AUC = {roc:.3f})')
ax.plot([0,1],[0,1],'--', color='gray', lw=1, label='Random')
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('SVM — ROC Curve\nAdult Census Dataset', fontsize=12)
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('output_plots/svm_02_roc_curve.png', dpi=150)
plt.show()

### 3.3 Performance Metrics Summary

In [ ]:
metrics = {
    'Accuracy':  acc,
    'Precision': precision_score(y_test, y_pred),
    'Recall':    recall_score(y_test, y_pred),
    'F1':        f1_score(y_test, y_pred),
    'ROC-AUC':  roc
}
fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(metrics.keys(), metrics.values(),
              color=['#4C72B0','#55A868','#C44E52','#8172B2','#937860'])
ax.set_ylim(0, 1.1)
for b in bars:
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.02,
            f'{b.get_height():.3f}', ha='center', va='bottom', fontsize=10)
ax.set_title('SVM — Performance Metrics\nAdult Census Dataset', fontsize=12)
ax.set_ylabel('Score'); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('output_plots/svm_03_metrics_bar.png', dpi=150)
plt.show()

## 4. Results Summary

| Metric | Score |
|--------|-------|
| Accuracy | 0.7950 |
| ROC-AUC | 0.9019 |
| Precision (>50K) | 0.5600 |
| Recall (>50K) | 0.8600 |
| F1 (>50K) | 0.6800 |

### Observations
- **ROC-AUC of 0.90** indicates strong discriminative ability despite class imbalance.
- `class_weight='balanced'` substantially improves recall on the minority class (>50K).
- Precision–recall trade-off is visible: high recall (0.86) comes at the cost of lower precision (0.56).
- StandardScaler is essential — SVM is sensitive to feature scale; without it, performance degrades significantly.

### Notes for Report Integration
- All plots saved to `output_plots/` with `svm_` prefix.
- Shared train/test split used — results are directly comparable with other group models.
- Model trained with `random_state=42` for reproducibility.